# Example 16.2 / 1: Solve a Grid With alldiff

Companion notebook for **16.2 Sudoku Solver** in the AMPL lab Documentation, section "Solve a Grid With `alldiff`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Grid With `alldiff`

Let $G=b^2$, let $v_{rc}\in\{1,\ldots,G\}$ be the digit in row $r$, column $c$, and let $a_{rc}$ contain a clue or zero for an empty cell. Then impose pairwise distinct values in every row, column, and subgrid, plus $v_{rc}=a_{rc}$ wherever a clue is present.

Because each group has `GridSize` integer variables, each bounded from 1 through `GridSize`, `alldiff` makes the group a permutation of the digits. No objective is needed.

With `Base = 2` the model solves a 4 × 4 grid with 2 × 2 boxes. The data sets `Base` and the keyed clues with `ampl.param[...]`; the run solves with HiGHS, checks `ampl.solve_result`, and reads `Digit` with `to_dict()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param Base integer > 0 default 3;  # box side b; the grid is Base^2 x Base^2
param GridSize := Base^2;  # digits per row, column and box

set ROWS := 1..GridSize;  # grid rows
set COLS := 1..GridSize;  # grid columns
set BOX {br in 1..Base, bc in 1..Base} =  # cells of the box in box row br, box column bc
    {(br-1)*Base+1..br*Base, (bc-1)*Base+1..bc*Base};

param Given {ROWS, COLS} integer >= 0, <= GridSize default 0;  # clue digit; 0 is an empty cell
var Digit {ROWS, COLS} integer >= 1, <= GridSize;  # digit placed in each cell

subject to RowDifferent {r in ROWS}:  # each digit once per row
    alldiff {c in COLS} Digit[r,c];
subject to ColumnDifferent {c in COLS}:  # each digit once per column
    alldiff {r in ROWS} Digit[r,c];
subject to BoxDifferent {br in 1..Base, bc in 1..Base}:  # each digit once per box
    alldiff {(r,c) in BOX[br,bc]} Digit[r,c];
subject to KeepGiven {r in ROWS, c in COLS: Given[r,c] > 0}:  # clues stay fixed
    Digit[r,c] = Given[r,c];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["Base"] = 2
clues = {(2, 3): 2, (2, 4): 1, (3, 2): 2, (3, 4): 4, (4, 2): 4, (4, 3): 1}
ampl.param["Given"] = clues

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
grid = ampl.var["Digit"].to_dict()
for r in range(1, 5):
    print(" ".join(str(round(grid[r, c])) for c in range(1, 5)))

solved
2 1 4 3
4 3 2 1
1 2 3 4
3 4 1 2


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; a zero in the `Given` table marks an empty cell:

In [6]:
%%writefile data.dat
param Base := 2;
param Given:  1 2 3 4 :=
          1   0 0 0 0
          2   0 0 2 1
          3   0 2 0 4
          4   0 4 1 0;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result;
display Digit;

solve_result = solved

Digit :=
1 1   2
1 2   1
1 3   4
1 4   3
2 1   4
2 2   3
2 3   2
2 4   1
3 1   1
3 2   2
3 3   3
3 4   4
4 1   3
4 2   4
4 3   1
4 4   2
;



### Another way in amplpy

Every clue reappears in the completed grid, which is the effect of `KeepGiven`.

The run below, with the same model.mod and data.py, prints `GridSize` and one box with `ampl.param[...].value()` and `ampl.set[...].to_list()`, then tries three bad clue sets. `ampl.param["Given"]` is given every cell, with zeros for empty ones, so no clue from an earlier solve is left behind:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["Base"] = 2
clues = {(2, 3): 2, (2, 4): 1, (3, 2): 2, (3, 4): 4, (4, 2): 4, (4, 3): 1}
ampl.param["Given"] = clues

In [11]:
# run.py
print("GridSize:", ampl.param["GridSize"].value())
print("BOX[1,2]:", ampl.set["BOX"][1, 2].to_list())
bad_clues = {
    "2 twice in row 1": {(1, 1): 2, (1, 2): 2},
    "no repeated clue": {(1, 2): 2, (1, 3): 3, (2, 2): 1, (3, 1): 4},
    "digit 5 in a 4 x 4 grid": {(1, 1): 5},
}
for label, bad in bad_clues.items():
    try:
        ampl.param["Given"] = {
            (r, c): bad.get((r, c), 0) for r in range(1, 5) for c in range(1, 5)
        }
        ampl.solve(solver="highs", verbose=False)
        print(f"{label}: {ampl.solve_result}")
    except Exception as error:
        print(f"{label}: {error}")

GridSize: 4
BOX[1,2]: [(1, 3), (1, 4), (2, 3), (2, 4)]
2 twice in row 1: infeasible
no repeated clue: infeasible
digit 5 in a 4 x 4 grid: Error executing "let" command:
error processing param Given[1,1]:
	failed check: param Given[1,1] = 5
		is not <= 4
		-- not <= GridSize;
